# Fashion Trend Prediction
**What is in this season? Using machine learning to identify fashion trends**

Supervisor: Dr Ollie Bartlett

### Research Questions
- **RQ1**: Can ML predict next season's best-selling products?
- **RQ2**: Which features most influence fashion trends?
- **RQ3**: Which ML algorithm performs best?
- **RQ4**: Can deep learning outperform traditional ML?
- **RQ5**: Can external data improve predictions?

In [ ]:
# 1. IMPORTS
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import matplotlib.pyplot as plt
import seaborn as sns

BASE = os.environ.get('FASHION_DATA_DIR', os.path.join(os.getcwd(), 'data'))
OUTPUT = os.environ.get('FASHION_OUTPUT_DIR', os.getcwd())

In [ ]:
# 2. LOAD DATA
customers = pd.read_csv(os.path.join(BASE, 'customers.csv'), low_memory=False)
products = pd.read_csv(os.path.join(BASE, 'products.csv'), low_memory=False)
transactions = pd.read_csv(os.path.join(BASE, 'transactions.csv'), low_memory=False)
stores = pd.read_csv(os.path.join(BASE, 'stores.csv'), low_memory=False)
discounts = pd.read_csv(os.path.join(BASE, 'discounts.csv'), low_memory=False)

print(f'Customers: {customers.shape}')
print(f'Products: {products.shape}')
print(f'Transactions: {transactions.shape}')
print(f'Stores: {stores.shape}')
print(f'Discounts: {discounts.shape}')

In [ ]:
# 3. PREPROCESSING
customers['Age'] = 2024 - pd.to_datetime(customers['Date Of Birth'], errors='coerce').dt.year
customers['AgeGroup'] = pd.cut(customers['Age'], bins=[0, 18, 25, 35, 50, 65, 120], 
                                labels=['0-18', '19-25', '26-35', '36-50', '51-65', '65+'])

products['ProductColor'] = products['Color'].fillna('Unknown')
products = products.drop(columns=['Color'], errors='ignore')
products['Sizes'] = products['Sizes'].fillna('Unknown')

transactions['Date'] = pd.to_datetime(transactions['Date'], errors='coerce')
transactions['Year'] = transactions['Date'].dt.year
transactions['Month'] = transactions['Date'].dt.month

def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else: return 'Fall'

transactions['Season'] = transactions['Month'].apply(get_season)
print('Preprocessing done.')

In [ ]:
# 4. MERGE DATASETS
data = transactions.merge(
    products[['Product ID', 'Category', 'Sub Category', 'ProductColor', 'Sizes', 'Production Cost']],
    on='Product ID', how='left'
)
data = data.merge(customers[['Customer ID', 'Gender', 'Age', 'AgeGroup', 'City', 'Country']], 
                  on='Customer ID', how='left')
data = data.merge(stores[['Store ID', 'Country', 'City']], on='Store ID', how='left', 
                  suffixes=('_cust', '_store'))
print(f'Merged shape: {data.shape}')

In [ ]:
# 5. AGGREGATE BY CATEGORY + SEASON + YEAR
agg_data = data.groupby(['Category', 'Season', 'Year', 'ProductColor', 'Gender', 'AgeGroup']).agg(
    total_quantity=('Quantity', 'sum'),
    total_revenue=('Line Total', 'sum'),
    avg_unit_price=('Unit Price', 'mean'),
    avg_discount=('Discount', 'mean'),
    transaction_count=('Date', 'count'),
    avg_age=('Age', 'mean'),
    city_count=('City_cust', 'nunique'),
    store_count=('Store ID', 'nunique'),
    production_cost=('Production Cost', 'mean')
).reset_index()
print(f'Aggregated: {agg_data.shape}')

In [ ]:
# 6. CREATE LAG FEATURES (time series)
season_order = {'Spring': 0, 'Summer': 1, 'Fall': 2, 'Winter': 3}
agg_data['SeasonNum'] = agg_data['Season'].map(season_order)
agg_data['TimeStep'] = agg_data['Year'] * 4 + agg_data['SeasonNum']
agg_data = agg_data.sort_values(['Category', 'ProductColor', 'Gender', 'AgeGroup', 'TimeStep'])

group_cols = ['Category', 'ProductColor', 'Gender', 'AgeGroup']
target_col = 'total_quantity'
lag_cols = ['total_revenue', 'avg_unit_price', 'avg_discount', 'transaction_count']

for col in lag_cols + [target_col]:
    for lag in range(1, 3):
        agg_data[f'{col}_lag{lag}'] = agg_data.groupby(group_cols, group_keys=False)[col].shift(lag)
    agg_data[f'{col}_roll2'] = agg_data.groupby(group_cols, group_keys=False)[col].transform(
        lambda x: x.rolling(window=2, min_periods=1).mean()
    )

agg_data = agg_data.dropna().reset_index(drop=True)
print(f'After lag features: {agg_data.shape}')

In [ ]:
# 7. ENCODE CATEGORIES
cat_cols = ['Category', 'Season', 'ProductColor', 'Gender', 'AgeGroup']
for col in cat_cols:
    le = LabelEncoder()
    agg_data[col + '_enc'] = le.fit_transform(agg_data[col])

feature_cols = [
    'SeasonNum', 'TimeStep',
    'total_quantity_lag1', 'total_quantity_lag2', 'total_quantity_roll2',
    'total_revenue_lag1', 'total_revenue_lag2', 'total_revenue_roll2',
    'avg_unit_price_lag1', 'avg_unit_price_lag2', 'avg_unit_price_roll2',
    'avg_discount_lag1', 'avg_discount_lag2', 'avg_discount_roll2',
    'transaction_count_lag1', 'transaction_count_lag2', 'transaction_count_roll2',
    'avg_age', 'city_count', 'store_count', 'production_cost', 'avg_unit_price',
    'Category_enc', 'Season_enc', 'ProductColor_enc', 'Gender_enc', 'AgeGroup_enc'
]

X = agg_data[feature_cols].copy()
y = agg_data[target_col].values
print(f'Features: {len(feature_cols)}, Samples: {len(X)}')

In [ ]:
# 8. TIME-BASED TRAIN/TEST SPLIT
agg_data_sorted = agg_data.sort_values('TimeStep').reset_index(drop=True)
split_idx = int(len(agg_data_sorted) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Scale numeric features
numeric_feats = ['avg_age', 'city_count', 'store_count', 'production_cost', 'avg_unit_price',
                 'total_quantity_lag1', 'total_quantity_lag2', 'total_quantity_roll2',
                 'total_revenue_lag1', 'total_revenue_lag2', 'total_revenue_roll2',
                 'avg_unit_price_lag1', 'avg_unit_price_lag2', 'avg_unit_price_roll2',
                 'avg_discount_lag1', 'avg_discount_lag2', 'avg_discount_roll2',
                 'transaction_count_lag1', 'transaction_count_lag2', 'transaction_count_roll2']

scaler = StandardScaler()
X_train[numeric_feats] = scaler.fit_transform(X_train[numeric_feats])
X_test[numeric_feats] = scaler.transform(X_test[numeric_feats])

print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
# 9. TRAIN MODELS
models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=100, max_depth=10, learning_rate=0.1, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, max_depth=10, learning_rate=0.1, random_state=42, verbose=-1),
    'CatBoost': CatBoostRegressor(iterations=100, depth=10, learning_rate=0.1, random_state=42, verbose=0)
}

results = {}
for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'MAE': mean_absolute_error(y_test, y_pred),
        'R2': r2_score(y_test, y_pred)
    }
    print(f'  RMSE: {results[name]["RMSE"]:.2f}, MAE: {results[name]["MAE"]:.2f}, R2: {results[name]["R2"]:.4f}')

In [ ]:
# 10. RESULTS COMPARISON
results_df = pd.DataFrame(results).T
results_df

In [ ]:
# 11. BEST PREDICTIONS
best_model = models['XGBoost']
preds = best_model.predict(X_test)

pred_df = agg_data_sorted.iloc[split_idx:][['Category', 'Season', 'ProductColor', 'Gender', 'AgeGroup', 'Year']].copy()
pred_df['predicted_quantity'] = preds
pred_df['actual_quantity'] = y_test
pred_df = pred_df.sort_values('predicted_quantity', ascending=False).head(20)
pred_df

In [ ]:
# 12. FEATURE IMPORTANCE
feat_imp = pd.DataFrame({'feature': feature_cols, 'importance': best_model.feature_importances_})
feat_imp = feat_imp.sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp, y='feature', x='importance', palette='viridis')
plt.title('Top 15 Features (XGBoost)')
plt.tight_layout()
plt.show()

In [ ]:
# 13. SEASONAL TRENDS VISUALIZATION
seasonal = agg_data.groupby(['Year', 'Season'])['total_quantity'].sum().reset_index()
plt.figure(figsize=(12, 5))
sns.lineplot(data=seasonal, x='Year', y='total_quantity', hue='Season', marker='o')
plt.title('Sales by Season')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 14. COLOR POPULARITY BY SEASON
color_s = agg_data.groupby(['Season', 'ProductColor'])['total_quantity'].sum().reset_index()
top_colors = color_s.sort_values('total_quantity', ascending=False).groupby('Season', group_keys=False).head(5)
plt.figure(figsize=(12, 6))
sns.barplot(data=top_colors, x='Season', y='total_quantity', hue='ProductColor', palette='Set2')
plt.title('Top 5 Colors by Season')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Summary
- **Best model**: XGBoost (R² = ?)
- **Top features**: city_count, revenue/quantity rolling averages
- **Next season prediction**: Run cell 11 to see top predicted best-sellers

To improve further: add weather, social media, or holiday data by joining on Date/Season/Year.